In [0]:
# Cell 1 — Create Sales Dataset
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import date, timedelta

spark = SparkSession.builder.getOrCreate()

# Seed data
regions     = ["North", "South", "East", "West", "Central"]
categories  = ["Electronics", "Apparel", "Furniture", "Groceries", "Sports"]
channels    = ["Online", "In-Store", "Partner"]
reps        = ["Arjun", "Priya", "Ravi", "Sneha", "Karthik",
               "Divya", "Ramesh", "Meena", "Suresh", "Lakshmi"]
table = "dashboardproject.sales.sales_data"
random.seed(42)
rows = []
base = date(2023, 1, 1)

for i in range(5000):
    sale_date  = base + timedelta(days=random.randint(0, 364))
    region     = random.choice(regions)
    category   = random.choice(categories)
    channel    = random.choice(channels)
    rep        = random.choice(reps)
    units      = random.randint(1, 50)
    unit_price = random.uniform(100, 5000)
    discount   = random.uniform(0, 0.3)
    revenue    = units * unit_price * (1 - discount)
    rows.append((i+1, sale_date.isoformat(), region, category,
                 channel, rep, units, unit_price, discount, revenue))

schema = StructType([
    StructField("sale_id",    IntegerType()),
    StructField("sale_date",  StringType()),
    StructField("region",     StringType()),
    StructField("category",   StringType()),
    StructField("channel",    StringType()),
    StructField("sales_rep",  StringType()),
    StructField("units_sold", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("discount",   DoubleType()),
    StructField("revenue",    DoubleType()),
])

df = spark.createDataFrame(rows, schema) \
          .withColumn("sale_date", to_date("sale_date"))

df.write.format("delta").mode("overwrite") \
        .saveAsTable(table)

print("Table created:", df.count(), "rows")
display(df.limit(10))

In [0]:
# Cell 2 — Quick sanity check queries (show to students)
spark.sql(f"""
  SELECT region,
         COUNT(*)          AS total_orders,
         ROUND(SUM(revenue),2) AS total_revenue,
         ROUND(AVG(revenue),2) AS avg_order_value
  FROM {table}
  GROUP BY region
  ORDER BY total_revenue DESC
""").display()